# Local AI Sales Intelligence

A complete source walkthrough in ten chapters. Each chapter contains explanations, full file-writing code cells, runnable fictional-data examples and a verification checkpoint.

Start Jupyter from this project using its virtual environment. Read `README.md` first. The delivered source files are already present. Source cells reconstruct a separate copy in `notebook_project/`; demonstrations import the supplied, tested package.

All demonstrations use clearly fictional transactions. Model and n8n calls are opt-in. `Run All` does not install system software, download models, publish workflows, or start services. To review one chapter at a time, use the individual notebooks and pause at each checkpoint.

The original prompt's ten topics are retained. Basic deterministic preparation comes before optional local models, and the complete web API is assembled after its modules exist.


In [ ]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the supplied project or its notebooks directory."
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Source-writing cells reconstruct a separate copy and preserve the supplied project.
BUILD_ROOT = ROOT / "notebook_project"
BUILD_ROOT.mkdir(exist_ok=True)
for folder in ["sales_intelligence", "scripts", "tests", "workflows", "frontend/src", "docs"]:
    (BUILD_ROOT / folder).mkdir(parents=True, exist_ok=True)
RUN_LOCAL_MODELS = False  # opt in only after installing local Ollama models
RUN_N8N = False          # opt in only after configuring/publishing the local workflows
print("Python:", sys.version.split()[0])
print("Source reconstruction:", BUILD_ROOT)


## Chapter 1 · Foundation

Create the project configuration, SQLite metadata store and bounded file reader. Original files stay separate from parsed and cleaned versions. The supplied implementation targets one local company with an authenticated API.

### Code cell · `pyproject.toml`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'pyproject.toml'}"
[build-system]
requires = ["hatchling>=1.26"]
build-backend = "hatchling.build"

[project]
name = "local-ai-sales-intelligence"
version = "0.1.0"
description = "Local, evidence-based sales analysis with a chapter-by-chapter notebook and n8n integration"
requires-python = ">=3.12,<3.15"
dependencies = [
  "fastapi>=0.115,<1", "uvicorn>=0.34,<1", "pydantic>=2.10,<3",
  "pydantic-settings>=2.7,<3", "python-multipart>=0.0.20,<1",
  "pandas>=2.2,<4", "numpy>=2,<3", "scipy>=1.14,<2", "statsmodels>=0.14,<1",
  "scikit-learn>=1.6,<2", "openpyxl>=3.1,<4", "xlrd>=2.0,<3", "olefile>=0.47,<1",
  "pyarrow>=19,<25", "plotly>=6,<7", "httpx>=0.28,<1",
  "langchain-ollama>=0.3,<2", "langgraph>=0.3,<2", "langgraph-checkpoint-sqlite>=2,<4",
  "faiss-cpu>=1.10,<2", "pypdf>=5,<7", "python-docx>=1.1,<2",
  "jdatetime>=5,<6", "psutil>=6,<8",
]

[project.optional-dependencies]
dev = ["pytest>=8,<10", "ruff>=0.11,<1", "nbformat>=5.10,<6", "nbclient>=0.10,<1", "jupyterlab>=4,<5", "ipykernel>=6,<8"]

[tool.pytest.ini_options]
testpaths = ["tests"]
filterwarnings = ["ignore:.*builtin type.*:DeprecationWarning"]

[tool.ruff]
line-length = 110
target-version = "py312"

[tool.hatch.build.targets.wheel]
packages = ["sales_intelligence"]


### Code cell · `.gitignore`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / '.gitignore'}"
.venv/
.env
.cache/
__pycache__/
.pytest_cache/
.ruff_cache/
.ipynb_checkpoints/
data/
notebook_data/
notebook_project/
frontend/node_modules/
frontend/dist/
*.egg-info/
*.zip


### Code cell · `.env.example`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / '.env.example'}"
# Create a private local .env with: python scripts/bootstrap.py
SALES_API_KEY=replace-with-a-long-random-value
SALES_DATA_DIR=./data
SALES_OLLAMA_URL=http://127.0.0.1:11434
SALES_LLM_MODEL=qwen3:8b
SALES_EMBEDDING_MODEL=nomic-embed-text
SALES_USE_LLM=false
SALES_MAX_UPLOAD_MB=25
SALES_MAX_ROWS=200000
SALES_MAX_COLUMNS=200
SALES_WORKERS=1
LANGSMITH_TRACING=false
LANGCHAIN_TRACING_V2=false
OLLAMA_NO_CLOUD=1


### Code cell · `sales_intelligence/__init__.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'sales_intelligence/__init__.py'}"
"""Local sales intelligence. All financial calculations run in Python."""

__version__ = "0.1.0"


### Code cell · `sales_intelligence/config.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'sales_intelligence/config.py'}"
from pathlib import Path
from urllib.parse import urlparse

from pydantic import Field, field_validator
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="SALES_", env_file=".env", extra="ignore")
    api_key: str = Field(min_length=24)
    data_dir: Path = Path("data")
    ollama_url: str = "http://127.0.0.1:11434"
    llm_model: str = "qwen3:8b"
    embedding_model: str = "nomic-embed-text"
    use_llm: bool = False
    max_upload_mb: int = Field(25, ge=1, le=100)
    max_rows: int = Field(200_000, ge=10, le=1_000_000)
    max_columns: int = Field(200, ge=3, le=1000)
    workers: int = Field(1, ge=1, le=2)
    llm_timeout: int = Field(120, ge=5, le=300)
    retention_days: int = Field(30, ge=1)

    @field_validator("ollama_url")
    @classmethod
    def local_only(cls, value):
        url = urlparse(value)
        if url.scheme != "http" or url.hostname not in {"localhost", "127.0.0.1", "::1", "ollama"}:
            raise ValueError("Ollama must use a loopback address or the local Compose service 'ollama'.")
        if url.username or url.password or url.query or url.fragment or url.path not in {"", "/"}:
            raise ValueError("Use a plain local Ollama origin.")
        return value.rstrip("/")

    @field_validator("llm_model", "embedding_model")
    @classmethod
    def no_cloud(cls, value):
        if not value or "cloud" in value.lower() or "/" in value:
            raise ValueError("Only an explicitly downloaded local model tag is allowed.")
        return value

    def prepare(self):
        self.data_dir = self.data_dir.resolve()
        self.data_dir.mkdir(parents=True, exist_ok=True, mode=0o700)
        return self


### Code cell · `sales_intelligence/schemas.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'sales_intelligence/schemas.py'}"
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, model_validator


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid", allow_inf_nan=False)


CANONICAL = {
    "customer_id",
    "customer_name",
    "product_id",
    "product_name",
    "order_id",
    "transaction_date",
    "sales_quantity",
    "sales_amount",
    "purchase_amount",
    "cost",
    "discount",
    "unit_price",
    "category",
    "region",
    "text",
    "currency",
}


class Mapping(StrictModel):
    sheet: str = Field(max_length=100)
    columns: dict[str, str]
    grain: Literal["order", "line_item"]
    currency: str = Field(min_length=1, max_length=20)
    sales_definition: str = Field(min_length=15, max_length=1000)
    sales_basis: Literal["net_after_discount", "gross_before_discount"] = "net_after_discount"
    cost_basis: Literal["unavailable", "line_total", "per_unit"] = "unavailable"
    date_calendar: Literal["gregorian", "jalali"] = "gregorian"
    date_format: str = "%Y-%m-%d"
    confirmed: bool = False

    @model_validator(mode="after")
    def validate_columns(self):
        if not self.confirmed:
            raise ValueError("A person must confirm column meanings and business definitions.")
        if set(self.columns) - CANONICAL:
            raise ValueError("Unknown canonical field.")
        if len(set(self.columns.values())) != len(self.columns):
            raise ValueError("One source column cannot represent two different fields.")
        if not {"transaction_date", "sales_amount"} <= self.columns.keys():
            raise ValueError("Map transaction_date and sales_amount.")
        if self.sales_basis == "gross_before_discount" and "discount" not in self.columns:
            raise ValueError("Gross sales require a mapped monetary discount column.")
        if self.cost_basis != "unavailable" and "cost" not in self.columns:
            raise ValueError("Map cost before enabling profitability.")
        if self.cost_basis == "per_unit" and "sales_quantity" not in self.columns:
            raise ValueError("Per-unit costs require quantity.")
        return self


class Relationship(StrictModel):
    sheet: str
    left_on: str
    right_on: str
    columns: list[str] = Field(min_length=1, max_length=30)
    cardinality: Literal["many_to_one", "one_to_one"] = "many_to_one"
    allow_unmatched: bool = False
    confirmed: bool = False


class Cleaning(StrictModel):
    numeric_nulls: Literal["preserve", "zero", "error"] = "preserve"
    invalid_values: Literal["error", "preserve_as_null"] = "error"
    duplicates: Literal["keep", "drop_exact"] = "keep"
    decimal_separator: Literal[".", ","] = "."
    thousands_separator: Literal[",", ".", "", "٬"] = ","
    confirmed: bool = False

    @model_validator(mode="after")
    def validate_policy(self):
        if not self.confirmed:
            raise ValueError("Confirm the cleaning policy before any values change.")
        if self.decimal_separator == self.thousands_separator:
            raise ValueError("Decimal and thousands separators must differ.")
        return self


class Filters(StrictModel):
    start: str | None = None
    end: str | None = None
    customer_id: str | None = Field(None, max_length=200)
    product_id: str | None = Field(None, max_length=200)
    limit: int = Field(20, ge=1, le=100)


class Chat(StrictModel):
    question: str = Field(min_length=1, max_length=2000)
    filters: Filters = Field(default_factory=Filters)
    use_llm: bool = False


class Forecast(StrictModel):
    horizon: int = Field(7, ge=1, le=30)
    missing_days: Literal["error", "zero_confirmed"] = "error"


class TextClassifier(StrictModel):
    purpose: Literal["sentiment", "complaint_classification"]
    label_column: str
    label_definition: str = Field(min_length=20, max_length=1000)
    permitted_labels: list[str] = Field(min_length=2, max_length=30)
    labeling_reviewed: bool = False


class Supervised(StrictModel):
    task: Literal["classification", "regression"]
    target: str
    features: list[str] = Field(min_length=1, max_length=30)
    target_definition: str = Field(min_length=20, max_length=1000)
    prediction_horizon: str = Field(min_length=3, max_length=200)
    leakage_reviewed: bool = False


class Simulation(StrictModel):
    price_change: float = Field(0, ge=-0.9, le=2)
    volume_change: float = Field(0, ge=-1, le=2)
    discount_change_points: float = Field(0, ge=-1, le=1)
    current_discount_rate: float = Field(0, ge=0, lt=1)
    assumptions_confirmed: bool = False


class Route(StrictModel):
    tool: Literal[
        "summary", "products", "customers", "trend", "rfm", "retention", "basket", "forecast", "unsupported"
    ]


class AdviceItem(StrictModel):
    action: Literal[
        "review_product_mix", "investigate_trend", "review_customer_followup", "verify_data_quality"
    ]
    evidence_ids: list[str] = Field(min_length=1, max_length=5)


class Advice(StrictModel):
    items: list[AdviceItem] = Field(default_factory=list, max_length=5)


### Code cell · `sales_intelligence/store.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'sales_intelligence/store.py'}"
import hashlib
import json
import sqlite3
from contextlib import contextmanager
from datetime import UTC, datetime
from pathlib import Path
from uuid import UUID, uuid4


def now():
    return datetime.now(UTC).isoformat()


def dumps(value):
    return json.dumps(value, ensure_ascii=False, allow_nan=False, default=str, sort_keys=True)


class Store:
    def __init__(self, root: Path):
        self.root = Path(root).resolve()
        self.root.mkdir(parents=True, exist_ok=True, mode=0o700)
        with self.db() as db:
            db.executescript("""
            CREATE TABLE IF NOT EXISTS datasets (
                id TEXT PRIMARY KEY, created TEXT NOT NULL, updated TEXT NOT NULL,
                version INTEGER NOT NULL DEFAULT 0, meta TEXT NOT NULL);
            CREATE TABLE IF NOT EXISTS tasks (
                id TEXT PRIMARY KEY, dataset_id TEXT NOT NULL, kind TEXT NOT NULL,
                status TEXT NOT NULL, progress INTEGER NOT NULL DEFAULT 0,
                created TEXT NOT NULL, updated TEXT NOT NULL, payload TEXT NOT NULL,
                result TEXT, error TEXT, cancel INTEGER NOT NULL DEFAULT 0,
                UNIQUE(dataset_id, kind, payload));
            CREATE TABLE IF NOT EXISTS audit (
                id INTEGER PRIMARY KEY, dataset_id TEXT NOT NULL,
                created TEXT NOT NULL, action TEXT NOT NULL, detail TEXT NOT NULL);
            CREATE TABLE IF NOT EXISTS prompts (
                name TEXT NOT NULL, hash TEXT NOT NULL, content TEXT NOT NULL,
                created TEXT NOT NULL, PRIMARY KEY(name, hash));
            """)

    @contextmanager
    def db(self):
        db = sqlite3.connect(self.root / "metadata.sqlite", timeout=30)
        db.row_factory = sqlite3.Row
        db.execute("PRAGMA journal_mode=WAL")
        try:
            yield db
            db.commit()
        finally:
            db.close()

    def directory(self, dataset_id):
        UUID(dataset_id)
        path = self.root / dataset_id
        if not path.is_dir():
            raise KeyError("Dataset does not exist.")
        return path

    def create(self, meta):
        dataset_id = str(uuid4())
        (self.root / dataset_id).mkdir(mode=0o700)
        with self.db() as db:
            db.execute("INSERT INTO datasets VALUES (?, ?, ?, 0, ?)", (dataset_id, now(), now(), dumps(meta)))
        return dataset_id

    def get(self, dataset_id):
        UUID(dataset_id)
        with self.db() as db:
            row = db.execute("SELECT * FROM datasets WHERE id=?", (dataset_id,)).fetchone()
        if row is None:
            raise KeyError("Dataset does not exist.")
        return {**dict(row), "meta": json.loads(row["meta"])}

    def update(self, dataset_id, meta, expected_version):
        with self.db() as db:
            count = db.execute(
                "UPDATE datasets SET meta=?, version=version+1, updated=? WHERE id=? AND version=?",
                (dumps(meta), now(), dataset_id, expected_version),
            ).rowcount
        if not count:
            raise ValueError("Dataset changed. Refresh before applying this operation.")
        return expected_version + 1

    def audit(self, dataset_id, action, detail):
        with self.db() as db:
            db.execute(
                "INSERT INTO audit(dataset_id,created,action,detail) VALUES (?,?,?,?)",
                (dataset_id, now(), action, dumps(detail)),
            )

    def cache_path(self, dataset_id, kind, parameters, version=None):
        version = self.get(dataset_id)["version"] if version is None else version
        key = hashlib.sha256(
            dumps(
                {
                    "version": version,
                    "parameters": parameters,
                    "analysis_version": "0.1.0",
                    "kind": kind,
                }
            ).encode()
        ).hexdigest()
        directory = self.directory(dataset_id) / "cache"
        directory.mkdir(exist_ok=True)
        return directory / f"{key}.json"

    def task(self, task_id):
        UUID(task_id)
        with self.db() as db:
            row = db.execute("SELECT * FROM tasks WHERE id=?", (task_id,)).fetchone()
        if row is None:
            raise KeyError("Task does not exist.")
        result = dict(row)
        for key in ["payload", "result"]:
            result[key] = json.loads(result[key]) if result[key] else None
        return result


### Code cell · `sales_intelligence/ingestion.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'sales_intelligence/ingestion.py'}"
"""Bounded parsing. Workbooks are data, never executable instructions."""

import hashlib
import io
import struct
import zipfile
from datetime import datetime
from pathlib import Path, PurePosixPath
from uuid import uuid4

import olefile
import openpyxl
import pandas as pd
import xlrd

from .store import dumps


def check_archive(body, maximum_uncompressed=150_000_000):
    try:
        with zipfile.ZipFile(io.BytesIO(body)) as archive:
            infos = archive.infolist()
            if len(infos) > 5000 or sum(i.file_size for i in infos) > maximum_uncompressed:
                raise ValueError("Workbook archive exceeds the expanded-size limit.")
            for item in infos:
                name = PurePosixPath(item.filename.replace("\\", "/"))
                if name.is_absolute() or ".." in name.parts or item.flag_bits & 1:
                    raise ValueError("Unsafe or encrypted archive entry.")
                if item.file_size > 1_000_000 and item.file_size / max(item.compress_size, 1) > 200:
                    raise ValueError("Suspicious compression ratio.")
                lower = str(name).lower()
                if any(
                    token in lower for token in ["vbaproject", "macrosheet", "externallinks", "embeddings/"]
                ):
                    raise ValueError("Macros, embedded objects and external workbook links are not accepted.")
    except zipfile.BadZipFile as exc:
        raise ValueError("Invalid XLSX archive.") from exc


def check_legacy(body):
    if not olefile.isOleFile(io.BytesIO(body)):
        raise ValueError("XLS must be an unencrypted OLE workbook.")
    with olefile.OleFileIO(io.BytesIO(body)) as ole:
        names = ["/".join(parts).lower() for parts in ole.listdir()]
        if any(any(marker in name for marker in ["vba", "macro", "objectpool"]) for name in names):
            raise ValueError("Legacy workbook contains macro or embedded object streams.")
        name = "Workbook" if ole.exists("Workbook") else "Book"
        stream = ole.openstream(name).read()
        offset = 0
        while offset + 4 <= len(stream):
            record, length = struct.unpack_from("<HH", stream, offset)
            content = stream[offset + 4 : offset + 4 + length]
            if record == 0x002F:
                raise ValueError("Encrypted XLS is not supported.")
            if record == 0x0085 and len(content) >= 6 and content[5] != 0:
                raise ValueError("Only ordinary worksheets are accepted in legacy XLS.")
            if record == 0x0006:
                raise ValueError("Export workbook formulas as values before uploading.")
            offset += 4 + length


def validate_headers(values, max_columns):
    headers = [str(value).strip() if value is not None else "" for value in values]
    if len(headers) > max_columns or not headers or any(not v for v in headers):
        raise ValueError("Every column needs a nonempty header; check the column limit.")
    if len(set(headers)) != len(headers):
        raise ValueError("Duplicate column headers must be renamed in the source file.")
    if any(h.startswith(("_", "raw__")) for h in headers):
        raise ValueError("Headers beginning with '_' or 'raw__' are reserved.")
    return headers


def read_tables(body, extension, settings):
    tables = {}
    remaining = settings.max_rows
    if extension == ".csv":
        text = body.decode("utf-8-sig", errors="strict")
        import csv

        try:
            dialect = csv.Sniffer().sniff(text[:8192], delimiters=",;\t|")
            delimiter = dialect.delimiter
        except csv.Error:
            delimiter = ","
        rows = csv.reader(io.StringIO(text), delimiter=delimiter)
        headers = validate_headers(next(rows, []), settings.max_columns)
        frame = pd.read_csv(
            io.StringIO(text),
            sep=delimiter,
            dtype="string",
            keep_default_na=False,
            nrows=remaining + 1,
            on_bad_lines="error",
        )
        frame.columns = headers
        tables["CSV"] = frame.replace("", pd.NA)
    elif extension == ".xlsx":
        check_archive(body)
        book = openpyxl.load_workbook(io.BytesIO(body), read_only=True, data_only=False, keep_links=False)
        try:
            if len(book.worksheets) > 30:
                raise ValueError("At most 30 worksheets are supported.")
            for sheet in book.worksheets:
                if sheet.max_row > remaining + 1 or sheet.max_column > settings.max_columns:
                    raise ValueError("Worksheet exceeds row or column limits.")
                rows = sheet.iter_rows()
                first = next(rows, None)
                if first is None or all(c.value is None for c in first):
                    continue
                if any(cell.data_type == "f" for cell in first):
                    raise ValueError("Column headers cannot contain formulas.")
                headers = validate_headers([cell.value for cell in first], settings.max_columns)
                values = []
                for cells in rows:
                    if any(cell.data_type == "f" for cell in cells):
                        raise ValueError("Export formulas as values before uploading the workbook.")
                    if len(values) >= remaining:
                        raise ValueError("Workbook exceeds the total row limit.")
                    values.append(
                        [
                            cell.value.date().isoformat()
                            if isinstance(cell.value, datetime)
                            and cell.value.time().isoformat() == "00:00:00"
                            else cell.value
                            for cell in cells
                        ]
                    )
                remaining -= len(values)
                tables[sheet.title] = (
                    pd.DataFrame(values, columns=headers).astype("string").replace("", pd.NA)
                )
        finally:
            book.close()
    elif extension == ".xls":
        check_legacy(body)
        book = xlrd.open_workbook(file_contents=body, on_demand=True)
        try:
            if book.nsheets > 30:
                raise ValueError("At most 30 worksheets are supported.")
            for sheet in book.sheets():
                if sheet.nrows < 1:
                    continue
                if sheet.nrows - 1 > remaining or sheet.ncols > settings.max_columns:
                    raise ValueError("Workbook exceeds row or column limits.")
                headers = validate_headers(sheet.row_values(0), settings.max_columns)
                values = []
                for number in range(1, sheet.nrows):
                    row = []
                    for cell in sheet.row(number):
                        value = cell.value
                        if cell.ctype == xlrd.XL_CELL_DATE:
                            value = xlrd.xldate_as_datetime(value, book.datemode).isoformat(sep=" ")
                        elif cell.ctype == xlrd.XL_CELL_NUMBER and float(value).is_integer():
                            value = str(int(value))
                        row.append(value)
                    values.append(row)
                remaining -= len(values)
                tables[sheet.name] = pd.DataFrame(values, columns=headers).astype("string").replace("", pd.NA)
        finally:
            book.release_resources()
    else:
        raise ValueError("Supported file types: .csv, .xlsx and restricted .xls.")
    if not tables or any(len(frame) > settings.max_rows for frame in tables.values()):
        raise ValueError("Empty file or too many rows.")
    if sum(len(frame) for frame in tables.values()) == 0:
        raise ValueError("The file contains no data rows.")
    return tables


def ingest(store, settings, filename, body):
    if not body or len(body) > settings.max_upload_mb * 1024 * 1024:
        raise ValueError("Empty file or upload exceeds the configured size limit.")
    # Client filenames are metadata only; never concatenate them into storage paths.
    extension = Path(filename.replace("\\", "/")).suffix.lower()
    tables = read_tables(body, extension, settings)
    meta = {
        "filename": Path(filename.replace("\\", "/")).name[:200],
        "extension": extension,
        "content_hash": hashlib.sha256(body).hexdigest(),
        "state": "uploaded",
        "sheets": {},
    }
    dataset_id = store.create(meta)
    directory = store.directory(dataset_id)
    (directory / f"original{extension}").write_bytes(body)
    for index, (name, frame) in enumerate(tables.items()):
        path = f"sheet_{index}.parquet"
        frame.to_parquet(directory / path, index=False)
        meta["sheets"][name] = {"path": path, "rows": len(frame), "columns": list(frame.columns)}
    store.update(dataset_id, meta, 0)
    store.audit(dataset_id, "upload", {"content_hash": meta["content_hash"], "sheets": list(tables)})
    return store.get(dataset_id)


def raw_sheet(store, dataset_id, sheet):
    data = store.get(dataset_id)
    if sheet not in data["meta"]["sheets"]:
        raise ValueError("Unknown sheet.")
    return pd.read_parquet(store.directory(dataset_id) / data["meta"]["sheets"][sheet]["path"])


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_name(f".{path.name}.{uuid4().hex}.tmp")
    temporary.write_text(dumps(value), encoding="utf-8")
    temporary.replace(path)


### Code cell · `scripts/bootstrap.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'scripts/bootstrap.py'}"
"""Create local secrets once. Does not install software or start services."""

import os
import secrets
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
target = ROOT / ".env"
if target.exists():
    print("Existing .env preserved.")
else:
    text = (ROOT / ".env.example").read_text()
    text = text.replace("replace-with-a-long-random-value", secrets.token_urlsafe(36))
    text += f"\nN8N_ENCRYPTION_KEY={secrets.token_hex(32)}\n"
    text += "N8N_IMAGE=docker.n8n.io/n8nio/n8n:2.46.4\n"
    text += "OLLAMA_IMAGE=ollama/ollama:0.40.14\n"
    descriptor = os.open(target, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    with os.fdopen(descriptor, "w") as stream:
        stream.write(text)
    print("Created private .env. No secrets printed. Local model calls are disabled by default.")


### Code cell · `scripts/make_sample.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'scripts/make_sample.py'}"
"""Reproducible fictional transactions, never real customer records."""

from pathlib import Path

import numpy as np
import pandas as pd


def create_sample(directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(42)
    rows = []
    products = ["P01", "P02", "P03", "P04", "P05"]
    prices = {"P01": 120, "P02": 75, "P03": 95, "P04": 180, "P05": 55}
    number = 0
    for date in pd.date_range("2025-01-01", periods=180):
        for _ in range(int(rng.integers(3, 7))):
            number += 1
            customer = f"C{int(rng.integers(1, 61)):03d}"
            selected = rng.choice(products, size=int(rng.integers(1, 4)), replace=False)
            for product in selected:
                quantity = int(rng.integers(1, 5))
                rows.append(
                    {
                        "order_id": f"O{number:05d}",
                        "customer_id": customer,
                        "product_id": str(product),
                        "transaction_date": date.date().isoformat(),
                        "sales_quantity": quantity,
                        "sales_amount": quantity * prices[product],
                        "cost": quantity * prices[product] * 0.6,
                        "discount": 0,
                        "feedback": rng.choice(["کیفیت خوب بود", "تحویل دیر رسید", "بسته‌بندی مناسب بود"]),
                        "risk_label": int(customer in {"C001", "C002", "C003", "C004", "C005"}),
                    }
                )
    sales = pd.DataFrame(rows)
    dimensions = pd.DataFrame({"product_key": products, "category_name": ["A", "A", "B", "B", "C"]})
    sales.to_csv(directory / "sales.csv", index=False)
    with pd.ExcelWriter(directory / "sales.xlsx", engine="openpyxl") as writer:
        sales.to_excel(writer, sheet_name="Sales", index=False)
        dimensions.to_excel(writer, sheet_name="Products", index=False)
    (directory / "company_policy.txt").write_text(
        "FICTIONAL DEMONSTRATION POLICY. Review customer consent before promotional follow-up. "
        "Product bundles require a stock check and manager approval. This is not an actual company policy.",
        encoding="utf-8",
    )
    return sales


if __name__ == "__main__":
    frame = create_sample(Path(__file__).resolve().parents[1] / "sample_data")
    print(
        f"Created {len(frame)} fictional line items. risk_label is synthetic for tests only, not a churn target."
    )


### Run the chapter example

In [ ]:
from tempfile import TemporaryDirectory
from sales_intelligence.config import Settings
from sales_intelligence.store import Store
from sales_intelligence.ingestion import ingest
with TemporaryDirectory() as directory:
    settings = Settings(api_key="fictional-notebook-key-" + "x"*32, data_dir=directory, _env_file=None).prepare()
    store = Store(settings.data_dir)
    dataset = ingest(store, settings, "tiny.csv", b"date,amount\n2025-01-01,100\n")
    assert dataset["meta"]["sheets"]["CSV"]["rows"] == 1
    print(dataset["meta"])


### Review checkpoint

Check that the original file and parsed table exist independently. CSV input is UTF-8. Formula-bearing workbooks are rejected; export calculated values before uploading. Review limits before importing large files.

Review this result before moving to the next chapter.